# flow5 against AeroSandbox: where do the two models differ?

Both models describe the same airframe (the ADH planforms). The notebook goes down a ladder. Each step removes one
possible cause, so the remaining gap points to the next one. All numbers are computed here from the source tree. The
flow5 numbers are the DAVE-ML aero model of node 1.2 (from the flow5 sweep named in `tools/flow5/settings/currawong.yaml`). The flow5 T7 point is
an inviscid analysis. The drag polar comes from the viscous alpha polar of the same sweep.

Models: **flow5** (3D panel method, DAVE-ML build-up), **AeroBuildup** (AeroSandbox: sections from NeuralFoil plus
component corrections, no interference between components), **VLM** (AeroSandbox vortex lattice: inviscid, wings only).

In [ ]:
import sys, warnings
sys.path.insert(0, "..")                    # tools/aerosandbox/lab.py
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import lab

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
L = lab.load("currawong")                   # reads aircraft/currawong (read only)
lab.style()                                 # after the load: importing aerosandbox resets the matplotlib style
mrp, c_ref, s_ref = L.ref["xyz_ref"], L.ref["c_m"], L.ref["s_m2"]
cg = lab.rollup(L.masses)["cg"]
mac = lab.mac_geometry(L)
print(L)

from scipy.optimize import fsolve

K = L.flow5_const()                                   # CLa, Cma, CLde, Cmde, CD0, K, ... (per rad)
t7 = lab.t7_reference(L)
dt = lab.design_targets()
W_now, V_cr = L.spec["flight"]["weight_N"], L.spec["flight"]["speed_m_s"]
cond_cr = lab.condition(V_cr, W_now, s_ref, c_ref)
cond_t7 = lab.condition(t7["V"], t7["mass_kg"] * lab.G0, s_ref, c_ref)
WINGS = ["wing_main", "htail", "fin"]

## Step 1. Do the inputs agree?

In [ ]:
at, rb = dt["athena"], dt["reuben"]
def q_s(V, S): return 0.5 * 1.225 * V ** 2 * S
cols = {
    "flow5 T7 (old ADH)": dict(S=s_ref, b=L.ref["b_m"], MAC=c_ref, m=t7["mass_kg"], cgx=t7["cg"][0], V=t7["V"]),
    "ADH now": dict(S=s_ref, b=L.ref["b_m"], MAC=c_ref, m=lab.rollup(L.masses)["mass_kg"], cgx=cg[0], V=V_cr),
    "Athena sizing": dict(S=at["Wing area"], b=at["Wing span"], MAC=at["MAC length"], m=at["m_total"], cgx=np.nan, V=at["Cruise speed"]),
    "Sheet: Reuben flow5 file": dict(S=rb["Wing area"], b=rb["Wing span"], MAC=rb["MAC"], m=rb["Mass"], cgx=rb["CoG"], V=rb["Cruise Velocity"]),
}
inp = pd.DataFrame(cols)
inp.index = ["S [m2]", "b [m]", "MAC [m]", "mass [kg]", "CG x [m]", "speed [m/s]"]
inp.loc["AR"] = inp.loc["b [m]"] ** 2 / inp.loc["S [m2]"]
inp.loc["CL for level flight"] = inp.loc["mass [kg]"] * lab.G0 / (0.5 * 1.225 * inp.loc["speed [m/s]"] ** 2 * inp.loc["S [m2]"])
display(inp)
print(f"Athena states C_L {at['C_L']} and C_D {at['C_D']} (C_D_o {at['C_D_o']}, C_D_i {at['C_D_i']}).")
print(f"Reuben's flow5 row states CL {rb['CL']} at {rb['Cruise AoA']} deg and CD {rb['CD']}; level flight at its own mass, speed and area needs CL "
      f"{inp.loc['CL for level flight', 'Sheet: Reuben flow5 file']:.3f}.")

## Step 2. Same geometry, same point: lift and moment slopes

All models at the flow5 T7 point (speed and mass of the T7 analysis), moments about the flow5 MRP, elevator zero.
`NP` = neutral point. `alpha_L0` = angle of zero lift. Rows without the fuselage have none in the model.

In [ ]:
mrp = L.ref["xyz_ref"]
def sl(names, fus, method):
    return lab.slopes(lab.subset(L, names, fus, mrp), cond_t7, method)
rows = {
    "flow5 (DAVE-ML, T7)": dict(CLa=K["CLa"], Cma=K["Cma"], SM=-K["Cma"] / K["CLa"], CL0=K["CL0"],
                                NP_x=mrp[0] - K["Cma"] / K["CLa"] * c_ref, alpha_L0_deg=-np.degrees(K["CL0"] / K["CLa"])),
    "VLM wing": sl(["wing_main"], False, "vlm"),
    "VLM wing + tail": sl(["wing_main", "htail"], False, "vlm"),
    "AeroBuildup wing": sl(["wing_main"], False, "buildup"),
    "AeroBuildup wing + tail": sl(["wing_main", "htail"], False, "buildup"),
    "AeroBuildup full (with fin, fuselage)": sl(WINGS, True, "buildup"),
}
static = pd.DataFrame(rows).T
display(static)
S1 = static
tail_vlm = S1.loc["VLM wing + tail", "CLa"] - S1.loc["VLM wing", "CLa"]
tail_ab = S1.loc["AeroBuildup wing + tail", "CLa"] - S1.loc["AeroBuildup wing", "CLa"]
print(f"Tail lift slope (referred to S): VLM {tail_vlm:.2f}, AeroBuildup {tail_ab:.2f} per rad.")

What flow5 implies. flow5 gives two numbers for the pitch behaviour: CLa and Cma. With a wing (aerodynamic centre x_w) and a
tail (centre x_t from the elevator derivatives, `x_t - x_mrp = -Cmde/CLde * c`) these fix the lift slope of each part:

    CLa = CLa_w + a_t
    Cma = CLa_w (x_mrp - x_w)/c  -  a_t (x_t - x_mrp)/c

In [ ]:
x_w = S1.loc["VLM wing", "NP_x"]                      # wing aerodynamic centre (VLM = AeroBuildup for the wing alone)
x_t = mrp[0] - K["Cmde"] / K["CLde"] * c_ref
A = np.array([[1.0, 1.0], [(mrp[0] - x_w) / c_ref, -(x_t - mrp[0]) / c_ref]])
cla_w, a_t = np.linalg.solve(A, [K["CLa"], K["Cma"]])
AR = L.ref["b_m"] ** 2 / s_ref
theory = 2 * np.pi * AR / (2 + np.sqrt(4 + AR ** 2))     # lifting-line (Helmbold), straight wing, e = 1, no sweep
print(f"wing aerodynamic centre x {x_w:+.3f} m, tail centre from the elevator derivatives x {x_t:+.3f} m")
print(f"flow5 implied: wing CLa {cla_w:.2f}, tail {a_t:.2f} per rad")
print(f"VLM wing {S1.loc['VLM wing', 'CLa']:.2f}, AeroBuildup wing {S1.loc['AeroBuildup wing', 'CLa']:.2f}, "
      f"lifting-line limit for AR {AR:.1f}: {theory:.2f} per rad")
print(f"-> the wing slope that flow5 implies is {100 * (1 - cla_w / S1.loc['VLM wing', 'CLa']):.0f} % below the VLM wing.")

## Step 3. Lift, drag and moment over alpha (elevator 0, T7 speed)

In [ ]:
alphas = np.arange(-4, 12.1, 2.0)
vlm_p = lab.subset(L, WINGS, False, mrp); ab_p = lab.subset(L, WINGS, True, mrp)
data = {"flow5": [], "AeroBuildup": [], "VLM": []}
for a in alphas:
    o = L.flow5_eval(a); data["flow5"].append((o["CL"], o["CD"], o["Cm"]))
    r = lab.run_aero(ab_p, cond_t7, a, "buildup"); data["AeroBuildup"].append((r["CL"], r["CD"], r["Cm"]))
    r = lab.run_aero(vlm_p, cond_t7, a, "vlm"); data["VLM"].append((r["CL"], r["CD"], r["Cm"]))
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for k, v in data.items():
    v = np.array(v)
    ax[0].plot(alphas, v[:, 0], "o-", color=lab.COLORS[k], label=k, ms=4)
    ax[1].plot(v[:, 1], v[:, 0], "o-", color=lab.COLORS[k], label=k, ms=4)
    ax[2].plot(alphas, v[:, 2], "o-", color=lab.COLORS[k], label=k, ms=4)
ax[0].set(xlabel="alpha [deg]", ylabel="CL", title="Lift"); ax[1].set(xlabel="CD", ylabel="CL", title="Drag polar")
ax[2].set(xlabel="alpha [deg]", ylabel="Cm about the MRP", title="Pitching moment")
ax[1].text(0.02, 0.03, "VLM: induced drag only", transform=ax[1].transAxes, color="#52514e")
ax[0].scatter([t7["alpha_deg"]], [t7["CL"]], marker="x", color="black", zorder=5, label="flow5 T7 trim point"); ax[0].legend()
fig.tight_layout(); fig.savefig(L.out / "polars.png", dpi=150); plt.show()

## Step 4. Sections: does the camber lift explain the difference in CL0?

AeroBuildup takes the lift of each section from NeuralFoil (viscous, at the section Reynolds number). The VLM and flow5 T7
are inviscid. The angle of zero lift of the foils shows how much lift comes from camber.

In [ ]:
foils = {}
for s in L.spec["surfaces"]:
    foils.update(s["foils"])
al = np.arange(-10, 14.1, 1.0)
rows, fig, ax = [], *plt.subplots(1, 2, figsize=(11, 4))
palette = ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#52514e"]
for (name, xy), col in zip(foils.items(), palette):
    p = lab.foil_polar(name, xy, cond_cr["Re_c"], al)
    pre = al <= 6                                           # before the stall: CL rises with alpha
    cl0_a = float(np.interp(0.0, p["CL"][pre], al[pre]))
    lin = (al >= -2) & (al <= 4)
    slope = np.polyfit(al[lin], p["CL"][lin], 1)[0]
    rows.append({"foil": name, "alpha_L0 [deg]": cl0_a, "CLa 2D [per rad]": slope * 180 / np.pi, "CD min": p["CD"].min(),
                 "CL max": p["CL"].max()})
    ax[0].plot(al, p["CL"], color=col, label=name); ax[1].plot(p["CD"], p["CL"], color=col, label=name)
ax[0].set(xlabel="alpha [deg]", ylabel="CL", title=f"NeuralFoil sections, Re {cond_cr['Re_c']:.2e}"); ax[1].set(xlabel="CD", ylabel="CL")
ax[0].legend(fontsize=8)
display(pd.DataFrame(rows).set_index("foil"))
fig.tight_layout(); plt.show()
print(f"3D angle of zero lift: flow5 {S1.loc['flow5 (DAVE-ML, T7)', 'alpha_L0_deg']:.2f} deg, VLM wing {S1.loc['VLM wing', 'alpha_L0_deg']:.2f} deg, "
      f"AeroBuildup wing {S1.loc['AeroBuildup wing', 'alpha_L0_deg']:.2f} deg (the wing has washout: its root is at +4 deg twist).")

## Step 5. Trim in level flight (the cruise point)

Level flight at the cruise speed and the weight of the Master Specification Sheet mass model. Unknowns: alpha and elevator.
Moments about the CG. flow5 has its moments about the MRP (the T7 CoG), so its moment is moved to the CG with
`Cm' = Cm + CL (x_cg - x_mrp)/c`.

In [ ]:
cg_now, cg_t7 = lab.rollup(L.masses)["cg"], list(L.ref["xyz_ref"])
rows = {}
for lbl, xyz in (("CG of the sheet mass model", cg_now), ("T7 CoG (MRP)", cg_t7)):
    rows[f"AeroBuildup, {lbl}"] = lab.trim_buildup(lab.subset(L, WINGS, True, xyz), cond_cr)
    rows[f"flow5, {lbl}"] = lab.trim_flow5(L, cond_cr, xyz[0])
trim = pd.DataFrame(rows).T
display(trim)
print(f"Level flight: V {V_cr} m/s, weight {W_now:.2f} N, CL needed {cond_cr['CL_req']:.3f}, Re(c) {cond_cr['Re_c']:.2e}")

## Step 6. Drag build-up at the cruise point

AeroBuildup is run on each component alone at the trimmed alpha and elevator (no interference between components).
The VLM at the same lift gives the inviscid induced drag. flow5 gives `CD = CD0 + K CL^2 + CDde |de|`: its K includes the rise
of the profile drag with CL, because K is a fit to the viscous polar. `e` is the span efficiency that a K would mean if it were all induced.

In [ ]:
base = lab.subset(L, WINGS, True, cg_now)
tr = lab.trim_buildup(base, cond_cr)
alpha_tr, de_tr = tr["alpha_deg"], tr["de_deg"]
parts = {}
for nm, kw in (("wing", dict(wings=["wing_main"])), ("horizontal tail", dict(wings=["htail"])),
               ("fin", dict(wings=["fin"])), ("fuselage", dict(wings=[], fuselage=True))):
    parts[nm] = lab.run_aero(lab.subset(L, xyz_ref=cg_now, **kw), cond_cr, alpha_tr, "buildup", de_tr)["CD"]
ab_total = tr["CD"]
# inviscid induced drag at the same CL (VLM, wing + tail)
vp = lab.subset(L, ["wing_main", "htail", "fin"], False, cg_now)
aa = fsolve(lambda x: lab.run_aero(vp, cond_cr, x[0], "vlm")["CL"] - cond_cr["CL_req"], [2.0])[0]
cdi_vlm = lab.run_aero(vp, cond_cr, aa, "vlm")["CD"]
CL = cond_cr["CL_req"]
f5 = lab.trim_flow5(L, cond_cr, cg_now[0])
ar = L.ref["b_m"] ** 2 / s_ref
drag = pd.DataFrame({
    "CD": [*parts.values(), ab_total, cdi_vlm, K["CD0"], K["K"] * CL ** 2, f5["CD"], at["C_D_o"], at["C_D_i"], at["C_D"]],
}, index=[*(f"AeroBuildup {k}" for k in parts), "AeroBuildup total (trimmed)", "VLM induced (wing, tail)",
          "flow5 CD0", "flow5 K CL^2", "flow5 total (trimmed)", "Athena C_D_o", "Athena C_D_i", "Athena C_D"])
drag["CdS [m2]"] = drag.CD * s_ref
display(drag)
print(f"span efficiency: VLM e = {CL ** 2 / (np.pi * ar * cdi_vlm):.2f}; flow5 K {K['K']:.4f} would mean e = {1 / (np.pi * ar * K['K']):.2f}; Athena C_D_i would mean e = {CL ** 2 / (np.pi * ar * at['C_D_i']):.2f} (at CL {CL:.3f})")
print(f"Parasite drag: AeroBuildup {ab_total - cdi_vlm:.4f} (total less VLM induced), flow5 {K['CD0']:.4f}, Athena {at['C_D_o']:.4f}")

## Step 7. Gap budget: aerodynamic power at the cruise point

`P = D V = CD q S V` at the same weight, speed and area. The steps move from the flow5 value to the AeroBuildup value, then to
Athena. The sizes of the steps depend on the order, because the split of the AeroBuildup drag into profile and induced drag
comes from the VLM (an approximation: no trim drag or interference in the VLM). The last bar is a residual, not a model.

In [ ]:
qSV = cond_cr["q"] * s_ref * V_cr
f5_total = f5["CD"]
ab_para = ab_total - cdi_vlm
steps = [
    ("flow5 (trimmed)", f5_total, None),
    ("parasite: flow5 CD0 -> AeroBuildup", ab_para - K["CD0"], None),
    ("induced + profile rise: flow5 K CL^2 -> VLM induced", cdi_vlm - K["K"] * CL ** 2, None),
    ("trim and the rest (elevator, interference)", ab_total - (f5_total + (ab_para - K["CD0"]) + (cdi_vlm - K["K"] * CL ** 2)), None),
]
wf = pd.DataFrame([{"step": n, "dCD": v, "dP [W]": v * qSV} for n, v, _ in steps])
wf.loc[0, "dCD"] = f5_total; wf.loc[0, "dP [W]"] = f5_total * qSV
wf["cumulative P [W]"] = wf["dP [W]"].cumsum()
display(wf)
extra = at["C_D"] - ab_total
print(f"AeroBuildup total {ab_total * qSV:.1f} W; flow5 {f5_total * qSV:.1f} W; Athena C_D at the same weight: {at['C_D'] * qSV:.1f} W "
      f"(Athena minus AeroBuildup: {extra * qSV:.1f} W, CdS {extra * s_ref:.4f} m2 not in either model)")
fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(len(wf)); cum = wf["cumulative P [W]"].to_numpy(); d = wf["dP [W]"].to_numpy()
bottoms = np.concatenate([[0.0], cum[:-1]]); bottoms[0] = 0.0
ax.bar(x, d, bottom=bottoms, color=["#2a78d6"] + ["#eb6834" if v >= 0 else "#1baf7a" for v in d[1:]], width=0.55)
ax.bar([len(wf)], [ab_total * qSV], color="#eb6834", width=0.55)
ax.bar([len(wf) + 1], [at["C_D"] * qSV], color="#52514e", width=0.55)
tops = list(np.maximum(bottoms, bottoms + d)) + [ab_total * qSV, at["C_D"] * qSV]
for xi, (top, val) in enumerate(zip(tops, list(cum) + [ab_total * qSV, at["C_D"] * qSV])):
    ax.text(xi, top + 4, f"{val:.0f} W", ha="center", color="#52514e")
ax.set_xticks(range(len(wf) + 2), ["flow5", "parasite", "induced", "trim, rest", "AeroBuildup", "Athena C_D"])
ax.set_ylabel("aerodynamic power [W]"); ax.set_title("From flow5 to AeroBuildup to Athena, level flight at the cruise speed")
fig.savefig(L.out / "power_budget.png", dpi=150); plt.show()

## Findings

The cell below prints the numbers that the findings quote, so they stay true when the mass model or the geometry change.

In [ ]:
print("1. Inputs:  S", f"{s_ref:.4f}", "vs Athena", at["Wing area"], "and the sheet flow5 file", rb["Wing area"],
      "| CG x: T7", round(t7["cg"][0], 3), "now", round(cg_now[0], 3), "sheet flow5 file", rb["CoG"])
print("2. Lift slope of the wing alone: VLM", round(S1.loc["VLM wing", "CLa"], 2), "AeroBuildup", round(S1.loc["AeroBuildup wing", "CLa"], 2),
      "flow5 (implied)", round(cla_w, 2), "per rad")
print("   Tail slope: VLM", round(tail_vlm, 2), "AeroBuildup", round(tail_ab, 2), "flow5 (implied)", round(a_t, 2))
print("   Neutral point x [m]: flow5", round(S1.loc["flow5 (DAVE-ML, T7)", "NP_x"], 3), "AeroBuildup", round(S1.loc["AeroBuildup full (with fin, fuselage)", "NP_x"], 3),
      "VLM wing+tail", round(S1.loc["VLM wing + tail", "NP_x"], 3))
print("3. Zero-lift angle [deg]: flow5", round(S1.loc["flow5 (DAVE-ML, T7)", "alpha_L0_deg"], 2), "VLM", round(S1.loc["VLM wing + tail", "alpha_L0_deg"], 2),
      "AeroBuildup", round(S1.loc["AeroBuildup full (with fin, fuselage)", "alpha_L0_deg"], 2))
print("4. Trim alpha / elevator [deg] at the sheet CG:  AeroBuildup", round(trim.iloc[0].alpha_deg, 2), round(trim.iloc[0].de_deg, 2),
      "| flow5", round(trim.iloc[1].alpha_deg, 2), round(trim.iloc[1].de_deg, 2))
print("5. Aerodynamic power at cruise [W]: flow5", round(f5_total * qSV, 1), "AeroBuildup", round(ab_total * qSV, 1), "Athena C_D", round(at["C_D"] * qSV, 1))